# 01 - Análisis Exploratorio de Datos (EDA)

Este notebook realiza el análisis exploratorio inicial de los datasets de phishing (emails) y smishing (SMS).

## Contenido
1. Carga de datasets
2. Estadísticas básicas
3. Distribución de clases
4. Análisis de longitud de textos
5. Análisis de palabras frecuentes
6. Visualizaciones

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data import EmailLoader, SMSLoader
from src.utils.visualization import Visualizer

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Carga de Datasets

In [ ]:
# Cargar dataset de SMS
sms_loader = SMSLoader(data_dir='../data')

# Intentar cargar el dataset UCI
try:
    sms_df = sms_loader.load_uci_sms_spam()
    print(f"Dataset SMS cargado: {len(sms_df)} mensajes")
except FileNotFoundError:
    print("Dataset no encontrado. Usando datos de ejemplo.")
    from src.data.sms_loader import create_sample_dataset
    sms_df = create_sample_dataset()

sms_df.head()

In [ ]:
# Cargar dataset de Email
email_loader = EmailLoader(data_dir='../data')

# Usar datos de ejemplo si no hay datos reales
from src.data.email_loader import create_sample_dataset as create_email_sample
email_df = create_email_sample()
print(f"Dataset Email cargado: {len(email_df)} correos")
email_df.head()

## 2. Estadísticas Básicas

In [ ]:
# Estadísticas del dataset SMS
print("=" * 50)
print("ESTADÍSTICAS DEL DATASET SMS")
print("=" * 50)
stats = sms_loader.get_dataset_stats(sms_df)
for key, value in stats.items():
    print(f"{key}: {value}")

In [ ]:
# Estadísticas del dataset Email
print("=" * 50)
print("ESTADÍSTICAS DEL DATASET EMAIL")
print("=" * 50)
stats = email_loader.get_dataset_stats(email_df)
for key, value in stats.items():
    print(f"{key}: {value}")

## 3. Distribución de Clases

In [ ]:
# Visualizar distribución de clases para SMS
visualizer = Visualizer(output_dir='../reports/figures')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SMS
sms_counts = sms_df['label'].value_counts()
axes[0].pie(sms_counts, labels=['Legítimo', 'Smishing'], autopct='%1.1f%%',
            colors=['#28A745', '#DC3545'], startangle=90)
axes[0].set_title('Distribución de Clases - SMS')

# Email
email_counts = email_df['label'].value_counts()
axes[1].pie(email_counts, labels=['Legítimo', 'Phishing'], autopct='%1.1f%%',
            colors=['#28A745', '#DC3545'], startangle=90)
axes[1].set_title('Distribución de Clases - Email')

plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150)
plt.show()

## 4. Análisis de Longitud de Textos

In [ ]:
# Añadir columnas de longitud
sms_df['length'] = sms_df['body'].str.len()
sms_df['word_count'] = sms_df['body'].str.split().str.len()

# Estadísticas por clase
print("Longitud de mensajes SMS por clase:")
print(sms_df.groupby('label')['length'].describe())

In [ ]:
# Visualizar distribución de longitud
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de longitud
for label, name, color in [(0, 'Legítimo', '#28A745'), (1, 'Smishing', '#DC3545')]:
    subset = sms_df[sms_df['label'] == label]['length']
    axes[0].hist(subset, bins=50, alpha=0.6, label=name, color=color)

axes[0].set_xlabel('Longitud (caracteres)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Longitud de SMS')
axes[0].legend()

# Boxplot
sms_df.boxplot(column='length', by='label', ax=axes[1])
axes[1].set_xticklabels(['Legítimo', 'Smishing'])
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('Longitud (caracteres)')
axes[1].set_title('Boxplot de Longitud por Clase')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../reports/figures/length_distribution.png', dpi=150)
plt.show()

## 5. Análisis de Palabras Frecuentes

In [ ]:
def get_top_words(texts, n=20):
    """Obtener las n palabras más frecuentes."""
    all_words = ' '.join(texts.fillna('')).lower().split()
    word_counts = Counter(all_words)
    return word_counts.most_common(n)

# Palabras más frecuentes en mensajes legítimos
legitimate_words = get_top_words(sms_df[sms_df['label'] == 0]['body'])
print("Top 20 palabras en mensajes LEGÍTIMOS:")
for word, count in legitimate_words:
    print(f"  {word}: {count}")

In [ ]:
# Palabras más frecuentes en mensajes de smishing
smishing_words = get_top_words(sms_df[sms_df['label'] == 1]['body'])
print("Top 20 palabras en mensajes de SMISHING:")
for word, count in smishing_words:
    print(f"  {word}: {count}")

In [ ]:
# Visualizar palabras frecuentes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Legítimos
words, counts = zip(*legitimate_words[:15])
axes[0].barh(range(len(words)), counts, color='#28A745')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].invert_yaxis()
axes[0].set_xlabel('Frecuencia')
axes[0].set_title('Top 15 Palabras - Mensajes Legítimos')

# Smishing
words, counts = zip(*smishing_words[:15])
axes[1].barh(range(len(words)), counts, color='#DC3545')
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words)
axes[1].invert_yaxis()
axes[1].set_xlabel('Frecuencia')
axes[1].set_title('Top 15 Palabras - Mensajes Smishing')

plt.tight_layout()
plt.savefig('../reports/figures/top_words.png', dpi=150)
plt.show()

## 6. Word Clouds

In [ ]:
try:
    from wordcloud import WordCloud
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Word cloud para mensajes legítimos
    legitimate_text = ' '.join(sms_df[sms_df['label'] == 0]['body'].fillna(''))
    wc_legitimate = WordCloud(width=800, height=400, background_color='white',
                               colormap='Greens').generate(legitimate_text)
    axes[0].imshow(wc_legitimate, interpolation='bilinear')
    axes[0].axis('off')
    axes[0].set_title('Word Cloud - Mensajes Legítimos', fontsize=14)
    
    # Word cloud para smishing
    smishing_text = ' '.join(sms_df[sms_df['label'] == 1]['body'].fillna(''))
    wc_smishing = WordCloud(width=800, height=400, background_color='white',
                             colormap='Reds').generate(smishing_text)
    axes[1].imshow(wc_smishing, interpolation='bilinear')
    axes[1].axis('off')
    axes[1].set_title('Word Cloud - Mensajes Smishing', fontsize=14)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/wordclouds.png', dpi=150)
    plt.show()
    
except ImportError:
    print("WordCloud no instalado. Ejecutar: pip install wordcloud")

## 7. Resumen y Conclusiones

### Observaciones principales:

1. **Distribución de clases**: [Completar con observaciones]
2. **Longitud de mensajes**: Los mensajes de smishing tienden a ser más largos/cortos que los legítimos
3. **Vocabulario**: Palabras como "free", "win", "prize" son más frecuentes en smishing
4. **Patrones identificados**: [Completar con observaciones]

### Próximos pasos:
- Preprocesamiento de texto (notebook 02)
- Extracción de características (notebook 03)
- Entrenamiento de modelos (notebook 04)